# Parsing one page: Docling, PaddleOCR-VL and Qwen VL

Notebook demo of the AI2Future workshop *Can We Rely on Extracted Documents? Parsing Methods Compared*
(Filip Pavičić, Atomic Intelligence).

- **What:** one page of a synthetic lab report, followed through a parsing pipeline.
- **Live:** Docling, on the CPU.
- **Loaded:** PaddleOCR-VL and Qwen VL, which need a GPU; their outputs are in `data/cached/`.
- **Can be skipped:** sections 5 and 7.

<details><summary>What is where</summary>

Loading files and drawing are in `src/dudemo/`. Everything the demo is about is in the cells below.
</details>

## Setup

- **Colab:** the first cell clones the repository and installs the dependencies, in less than a minute.
- **Locally:** install as described in the README, then start Jupyter from the repository.
- **Before you need it:** run the first Docling cell in section 2 once; it downloads Docling's models.

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and Path.cwd().name != "document-understanding-demo":
    if not Path("document-understanding-demo").exists():
        !git clone --quiet --depth 1 https://github.com/Atomic-Intelligence/document-understanding-demo.git
    %cd document-understanding-demo
    # pip warns that google-colab pins an older requests than Docling needs; Colab works with the
    # newer one, so the output is shown only if the install fails.
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--progress-bar", "off", "-e", "."],
                         capture_output=True, text=True)
    if pip.returncode != 0:
        print(pip.stdout, pip.stderr)
        raise RuntimeError("Installing the dependencies failed; pip's output is above.")
    print("Dependencies installed.")

# The repository root, whether Jupyter was started in notebooks/ or in the root
ROOT = Path.cwd() if (Path.cwd() / "src" / "dudemo").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import difflib
import json
import re
import time

import pandas as pd
from PIL import Image, ImageDraw, ImageOps

from dudemo import data, quiet, show

quiet()   # no model-loading logs in the output

## 1. The page

### 1.1 · The report

- A synthetic Croatian blood analysis report.
- Patient fields, two tables of results, units, reference ranges, and a flag column (H high, L low).
- An image only, with no text layer, so every parser has to read the pixels.

**Question for the room:** what would a parser get wrong on this page?

In [ ]:
PAGE = "lab_report_1"
page = data.page_image(PAGE)
show.image(page, f"{PAGE}.png")

### 1.2 · The ground truth

A JSON file next to the page. Each region has:

- `type`: heading, paragraph, key-value section or table;
- `box`: `[x0, y0, x1, y1]`, in pixels from the top-left corner;
- its content: `text`, key-value `pairs`, or a table as `html`.

One region of each type:

In [ ]:
ground_truth = json.loads((data.DATA / "ground_truth" / f"{PAGE}.json").read_text(encoding="utf-8"))

seen = set()
for region in ground_truth["regions"]:
    if region["type"] not in seen:
        seen.add(region["type"])
        print(json.dumps(region, ensure_ascii=False)[:260], "\n")

### 1.3 · One record, many views

- The JSON file is the lossless record of the page.
- Every other format is generated from it, for one use:
  - an image with boxes, to check the layout;
  - Markdown, to read and see the whole page;
  - text chunks with their page and box, for a search index or a RAG system.

**Image with boxes:** one rectangle per region, in its type's colour.

In [ ]:
COLORS = {"heading": "#E07B39", "paragraph": "#7F95A6", "key_value_section": "#5E9E1F", "table": "#058ACC"}

drawn = page.copy()
pen = ImageDraw.Draw(drawn)
for region in ground_truth["regions"]:
    pen.rectangle(region["box"], outline=COLORS[region["type"]], width=4)
show.image(drawn, "The ground truth, drawn: heading orange, text grey, key-value green, table blue")

### 1.4 · Markdown

The regions in reading order, each in the usual Markdown form for its type. Rendered, it is the target
every parser below is held against.

In [ ]:
def table_to_markdown(rows):
    header, *body = rows
    lines = ["| " + " | ".join(header) + " |", "|" + " --- |" * len(header)]
    lines += ["| " + " | ".join(row) + " |" for row in body]
    return "\n".join(lines)


def to_markdown(regions):
    parts = []
    for region in regions:                      # the file's order is the reading order
        if region["type"] == "heading":
            parts.append("## " + region["text"])
        elif region["type"] == "key_value_section":
            parts.append("\n".join(f"**{key}** {value}  " for key, value in region["pairs"]))
        elif region["type"] == "table":
            parts.append(table_to_markdown(data.html_rows(region["html"])))
        else:
            parts.append(region["text"])
    return "\n\n".join(parts)


show.markdown(to_markdown(ground_truth["regions"]), "Target Markdown")

### 1.5 · Zoom

Check a value against the pixels: a row of the page, enlarged, under what the file holds for it.

- Watch these two rows: they come back in section 4.

In [ ]:
truth = data.ground_truth(PAGE)
for label in ["Leukociti (WBC)", "ALT (ALAT)"]:
    x0, y0, x1, y1 = data.row_box(PAGE, label)
    row = page.crop((x0, y0 - 2, x1, y1 + 3))
    _header, cells = show.find_row(truth.tables, label)
    show.image(row.resize((row.width * 2, row.height * 2)), "The file says:  " + "  |  ".join(cells))

## 2. Parse it with Docling

- A pipeline of separate models, one for each step on the slides: detection, recognition,
  transformation.
- No part of it generates text: it can misread, but it cannot invent.

### 2.1 · Settings

The settings of the evaluation, each commented with its step. The first run downloads the models.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (LayoutObjectDetectionOptions, PdfPipelineOptions,
                                                RapidOcrOptions, TableFormerMode, TableStructureOptions)
from docling.document_converter import DocumentConverter, ImageFormatOption, PdfFormatOption

options = PdfPipelineOptions(
    layout_options=LayoutObjectDetectionOptions.from_preset("layout_heron_default"),  # 1. detection
    do_ocr=True,
    ocr_options=RapidOcrOptions(backend="torch"),                                     # 2. recognition: text
    do_table_structure=True,
    table_structure_options=TableStructureOptions(mode=TableFormerMode.ACCURATE),     # 2. recognition: tables
    generate_page_images=True,                                                       # to draw the regions
)
converter = DocumentConverter(format_options={
    InputFormat.IMAGE: ImageFormatOption(pipeline_options=options),
    InputFormat.PDF: PdfFormatOption(pipeline_options=options),
})
converter.initialize_pipeline(InputFormat.IMAGE)   # loads the models
print("Docling is ready.")

### 2.2 · Run

Docling reads the page, on this machine's CPU.

In [ ]:
start = time.perf_counter()
doc = converter.convert(data.page_path(PAGE)).document
print(f"Docling read the page in {time.perf_counter() - start:.0f} seconds on this machine.")

### 2.3 · Detection

The regions Docling found, in the same colours as the ground truth.

- Each patient label and each value is its own text region.

In [ ]:
docling = data.from_docling(doc)
show.regions(docling, "Docling: regions by type")

### 2.4 · Recognition

The second table as it is on the page, then the rows and columns TableFormer made of it.

In [ ]:
show.image(page.crop(data.table_box(PAGE, "Glukoza (na tašte)", pad=6)), "On the page")
show.table(doc.tables[1].export_to_dataframe(doc=doc))

### 2.5 · Transformation

- Docling keeps its result as a DoclingDocument, its lossless record, like the JSON file in 1.2.
- The regions in 2.3 and this Markdown are two views of it.

In [ ]:
show.markdown(docling, "Docling's Markdown")

Reading the Markdown shows at once where Docling struggles:

- the patient fields come out as loose lines, not as pairs;
- the tables get an extra column, with each range split over two cells;
- 10⁹/L becomes 109/L.

## 3. The same page, three other parsers

- Their outputs were made on a GPU and are loaded from `data/cached/`.
- Each one: its regions on the page, next to its Markdown, rendered.

### 3.1 · PaddleOCR-VL

- The hybrid: a layout detector labels the blocks, then a small vision-language model reads each block
  on its own.
- It finds the header and the footer (teal), but leaves them out of its Markdown.
- Its tables have the right columns; the units come back as LaTeX.

In [ ]:
paddle = data.cached(PAGE, "paddleocr_vl")
show.parse(paddle)

### 3.2 · Qwen VL, general

- End to end: it reads the whole page in one pass and writes each block as a tag, with its type, its box
  as `<location>`, and its text. One long prompt asks for this format.
- This is the model as released. Look at its `<location>` tags: the boxes are not read from the page,
  and the tags are malformed.

In [ ]:
qwen_general = data.cached(PAGE, "qwen_base")
print(data.first_blocks(data.raw_text(PAGE, "qwen_base"), 2))
show.parse(qwen_general)

### 3.3 · Qwen VL, fine-tuned

- The same model and prompt after our fine-tuning; only the weights differ.
- Its boxes are positions on the page: the headings, the patient block and the tables each sit where
  they are.

In [ ]:
qwen_finetuned = data.cached(PAGE, "qwen_finetuned")
print(data.first_blocks(data.raw_text(PAGE, "qwen_finetuned"), 2))
show.parse(qwen_finetuned)

### 3.4 · Four parsers, one page

**Question for the room:** which one would you start with for documents like this one?

In [ ]:
show.side_by_side(docling, paddle, qwen_general, qwen_finetuned)

## 4. Three kinds of error

Each example: the row on the page, the row as the page has it, and the row as the parser wrote it.
Cells that are not on the page are red.

- **Visibly broken:** anyone reading the output sees it.
- **Structure wrong:** the text may be right, but the shape is not.
- **Silent errors:** a wrong value that is well-formed and looks right. This is the dangerous kind.

<details><summary>Where silent errors come from</summary>

A vision-language model can write a value that is not on the page; OCR can misread one character as
another valid one.
</details>

In [ ]:
outputs = {"docling": docling, **{model: data.cached(PAGE, model) for model in data.CACHED_MODELS}}
for example in data.examples():
    show.error(example, outputs[example["model"]], truth)

## 5. General and fine-tuned Qwen VL

- The same model, prompt and settings; only the weights differ.
- On a second report. Dashed red: a region of the page that the model missed.

*Can be skipped.*

In [ ]:
PAGE_2 = "lab_report_2"
show.side_by_side(data.cached(PAGE_2, "qwen_base"), data.cached(PAGE_2, "qwen_finetuned"),
                  missed_from=data.ground_truth(PAGE_2))

- Fine-tuning taught the model the format: regions in the right place, tables as tables.
- It did not make every value right: the doctor *Dr. Ivić* becomes *Dr. Tić*, a well-formed name that
  nothing in the output marks as wrong.

## 6. Checks after extraction

A check reads only the parser's output and sends what looks wrong to a person.
Three rules for lab results:

- the unit is a known unit;
- the range looks like a range;
- the H or L flag fits the result and the range.

In [ ]:
UNITS = {"10⁹/L", "10¹²/L", "g/L", "mmol/L", "µmol/L", "μmol/L", "U/L", "%", "fL", "pg", "L/L", "mg/L"}
NUMBER = r"\d+(?:,\d+)?"
RANGE = re.compile(rf"(?P<low>{NUMBER}) - (?P<high>{NUMBER})|(?P<sign>[<>]) (?P<limit>{NUMBER})")


def number(text):
    return float(text.replace(",", "."))


def unit_is_known(unit):
    return unit in UNITS


def range_looks_like_a_range(reference):
    return RANGE.fullmatch(reference) is not None


def flag_fits(result, reference, flag):
    """H means above the range, L below it. A row without a flag is not checked."""
    match = RANGE.fullmatch(reference)
    if flag not in ("H", "L") or not match or not re.fullmatch(NUMBER, result):
        return None
    value = number(result)
    if match["low"]:
        expected = "L" if value < number(match["low"]) else "H" if value > number(match["high"]) else ""
    elif match["sign"] == "<":
        expected = "H" if value > number(match["limit"]) else ""
    else:
        expected = "L" if value < number(match["limit"]) else ""
    return expected == flag

Every row of Docling's two tables, through the three rules.

- ✗ sends the row to a person; – means there was nothing to check.

<details><summary>How the range is read</summary>

The range is everything between the unit and the flag column, so a range split over two columns is
checked as Docling wrote it.
</details>

In [ ]:
MARK = {True: "✓", False: "✗", None: "–"}

rows = []
for table in docling.tables:
    header = table[0]
    result_column, unit_column = header.index("Rezultat"), header.index("Jedinica")
    for row in table[1:]:
        result, unit, flag = row[result_column], row[unit_column], row[-1]
        reference = " ".join(cell for cell in row[unit_column + 1:-1] if cell)
        rows.append({"Parameter": row[0], "Result": result, "Unit": unit, "Range": reference, "Flag": flag,
                     "unit is known": MARK[unit_is_known(unit)],
                     "range looks like a range": MARK[range_looks_like_a_range(reference)],
                     "flag fits": MARK[flag_fits(result, reference, flag)]})
show.table(pd.DataFrame(rows))

- The reversed sign in the ALT row is caught only because the page also prints an H, which no longer
  fits the limit.
- A rule cannot tell whether a name or a number is right. For that, compare with a second parser:
  where two parsers disagree, a person looks.

The patient fields, general Qwen VL against PaddleOCR-VL:

<details><summary>Why field names are matched loosely</summary>

Parsers misspell labels too (PaddleOCR-VL writes *Liječenik* for *Liječnik*), so each field is matched to
the closest name.
</details>

In [ ]:
general = dict(data.cached(PAGE, "qwen_base").pairs)
second = data.cached(PAGE, "paddleocr_vl").pairs

rows = []
for field, value in second:
    match = difflib.get_close_matches(field, general, n=1, cutoff=0.75)
    other = general[match[0]] if match else "(missing)"
    rows.append({"Field": field, "PaddleOCR-VL": value, "Qwen VL, general": other,
                 "same in both": MARK[other == value]})
show.table(pd.DataFrame(rows))

- The general model changed one digit of the registration number and left out the date of birth.

## 7. A table across pages

- Parsers read one page at a time.
- Here the report is cut through its first table, as if the table went on to a second page.
- The rows at the top of page 2 have no header, so the parser does not know the columns.

*Can be skipped.*

In [ ]:
table = data.table_box(PAGE, "Hematokrit (Hct)")          # the first results table, in page pixels
page_break = data.row_box(PAGE, "Hematokrit (Hct)")[3]    # the bottom of this row
image = data.page_image(PAGE)
end_of_page_1 = image.crop((table[0], table[1], table[2], page_break))
top_of_page_2 = image.crop((table[0], page_break, table[2], table[3]))
show.page_break(end_of_page_1, top_of_page_2)

In [ ]:
def read_with_docling(picture, name):
    """Docling reads files, so the picture is saved first, with white space around it as a page margin."""
    path = Path("outputs") / f"{name}.png"
    path.parent.mkdir(exist_ok=True)
    ImageOps.expand(picture, border=40, fill="white").save(path)
    return data.from_docling(converter.convert(path).document)


show.markdown(read_with_docling(top_of_page_2, "top_of_page_2"), "Docling on the top of page 2")

- The first row of data became the header.
- The repair: stack both parts into one image and run the same parser again. The header travels with
  the rows, and nothing is retrained.

In [ ]:
both_parts = Image.new("RGB", (end_of_page_1.width, end_of_page_1.height + top_of_page_2.height), "white")
both_parts.paste(end_of_page_1, (0, 0))
both_parts.paste(top_of_page_2, (0, end_of_page_1.height))
show.image(both_parts, "Both parts stacked")
show.markdown(read_with_docling(both_parts, "both_parts"), "Docling on both parts stacked")

## 8. Try your own document

- Set `MY_FILE` to the path of a PDF or an image, and run the cell.
- On Colab, upload the file with the file browser on the left; its path is `/content/<file name>`.

Links: [Docling](https://github.com/docling-project/docling) ·
[PaddleOCR-VL](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6) ·
[Qwen2.5-VL](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)

In [ ]:
MY_FILE = "path/to/your/document.pdf"

if Path(MY_FILE).exists():
    show.parse(data.from_docling(converter.convert(MY_FILE).document))
else:
    print("Set MY_FILE to the path of a PDF or an image, then run this cell again.")